In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable as T

In [0]:
%run /Workspace/fmcg_pipeline/setup_fmcg_data/utilities

In [0]:
dbutils.widgets.text("catalog", "lakehouse", "Catalog")
dbutils.widgets.text("datasource", "orders", "Datasource")
catlog =dbutils.widgets.get("catalog")
dataSource = dbutils.widgets.get("datasource")

In [0]:
bronze_table = f"{catlog}.{bronze_schema}.{dataSource}"
silver_table = f"{catlog}.{silver_schema}.{dataSource}"
gold_table = f"{catlog}.{gold_schema}.fact_{dataSource}"

## bronze layer

In [0]:
landing_path = f's3://fmcg-child-data/{dataSource}/landing/'
processed_path = f's3://fmcg-child-data/{dataSource}/processed/'
df_bronze = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load(landing_path) \
    .select("*", "_metadata.file_name", "_metadata.file_size") \
    .withColumn("timestamp", F.current_timestamp())

In [0]:
df_bronze_fixed = df_bronze.withColumn("product_id", F.col("product_id").cast("int")).withColumn("order_qty", F.col("order_qty").cast("double"))


## Bronze staging tables

In [0]:
df_bronze_fixed.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{bronze_schema}.staging{dataSource}")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8920780931501238>, line 1
----> 1 df_bronze_fixed.write.format("delta")\
      2 .mode("overwrite")\
      3 .option("delta.enableChangeDataFeed", "true")\
      4 .saveAsTable(f"{catlog}.{bronze_schema}.staging{dataSource}")

NameError: name 'df_bronze_fixed' is not defined

In [0]:
df_bronze_fixed.count()

321

## Moving files from landing to processed

In [0]:
files = dbutils.fs.ls(landing_path)
for f in files:
    if f.name.endswith(".csv"):
        dbutils.fs.mv(f.path, f"{processed_path}/{f.name}", True)

## Silver layer

###  Staging silver table

In [0]:
df_silver = spark.read.table(f"{catlog}.{bronze_schema}.staging{dataSource}")
display(df_silver)


order_id,order_placement_date,customer_id,product_id,order_qty,file_name,file_size,timestamp
FDEC83401502,"Tuesday, December 02, 2025",789401,25891203,256.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC83401502,"Tuesday, December 02, 2025",789401,25891502,218.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC83401502,"Tuesday, December 02, 2025",789401,25891403,280.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC83401502,"Tuesday, December 02, 2025",789401,25891201,262.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC83401502,"Tuesday, December 02, 2025",789401,25891203,256.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC83401502,"Tuesday, December 02, 2025",789401,25891403,280.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC84202603,"Tuesday, December 02, 2025",789202,25891502,218.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC84202603,2025/12/02,789202,25891403,267.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC84202603,"Tuesday, December 02, 2025",789202,25891601,69.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z
FDEC84202603,"Tuesday, December 02, 2025",789202,25891602,126.0,orders_2025_12_02.csv,19621,2026-03-04T22:28:20.671Z


In [0]:
#date transformation
df_silver = df_silver.select("order_id", "order_placement_date", "customer_id", "product_id", "order_qty")
df_silver = df_silver.withColumn("order_placement_date",
    F.regexp_replace("order_placement_date",  r"^[A-Za-z]+,\s*", "")
    
)
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy" , "MMMM dd, yyyy"]

df_silver = df_silver.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("order_placement_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy"),
        F.try_to_date(F.col("order_placement_date"), "MMMM dd, yyyy")
    )
)
#null transformation
df_silver = df_silver.filter(F.col("order_qty").isNotNull())
#remove illegable value from customer_id 
df_silver = df_silver.withColumn("customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
    .otherwise(F.lit("999999"))
    
)
#drop duplicates
df_silver = df_silver.dropDuplicates(["order_id","customer_id","order_placement_date","order_qty","product_id"])
#get product_code
df_product = spark.table("lakehouse.silver.products")
df_product = df_product.dropDuplicates(["product_id"])
df_joined = df_silver.join(df_product, "product_id", "left")
df_silver = df_joined.drop("product","category","file_name","file_size","timestamp","variant","division")
df_silver = df_silver.withColumnRenamed("order_placement_date", "date") \
                         .withColumnRenamed("order_qty", "sold_quantity") \
                         .withColumnRenamed("product_id", "product_id")


In [0]:
display(df_silver)

product_id,order_id,date,customer_id,sold_quantity,product_code
25891201,FDEC83401502,2025-12-02,789401,262.0,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef
25891101,FDEC83601603,2025-12-02,789601,396.0,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
25891101,FDEC85402503,2025-12-02,789402,466.0,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
25891602,FDEC84601602,2025-12-02,789601,61.0,1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da
25891402,FDEC84720403,2025-12-02,999999,416.0,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7
25891101,FDEC83903303,2025-12-02,999999,500.0,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
25891202,FDEC84520503,2025-12-02,789520,380.0,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef
25891503,FDEC83221503,2025-12-02,789221,229.0,cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff
25891102,FDEC83422102,2025-12-02,789422,305.0,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373
25891203,FDEC84220401,2025-12-02,789220,331.0,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef


In [0]:
df_silver.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{silver_schema}.staging{dataSource}")

Add new data to Fact silver table

In [0]:
df_silver.write.format("delta")\
.mode("append")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{silver_table}")

## Gold Layer

In [0]:
df_gold = spark.table("lakehouse.silver.stagingorders")
df_monthly = df_gold.withColumn("month_start", F.date_trunc("month", F.col("date")))

df_monthly = df_monthly.groupBy("month_start","customer_id","product_code").agg(F.sum("sold_quantity").alias("sold_quantity"))
df_monthly = df_monthly.withColumnRenamed("customer_id", "customer_code") \
                   .withColumnRenamed("month_start", "date") \
                   .withColumn("date", F.to_date("date"))
df_monthly = df_monthly.withColumn("sold_quantity", F.col("sold_quantity").cast("long"))
df_monthly.write.mode("append").saveAsTable(f"{gold_table}")